In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [10]:
import os
import random
import shutil

# Paths from your Drive
img_source = '/content/drive/MyDrive/SesameProject/images'
lbl_source = '/content/drive/MyDrive/SesameProject/labels'
base_dir = '/content/dataset'

# Create YOLO structure locally in Colab
for folder in ['train', 'val']:
    for sub in ['images', 'labels']:
        os.makedirs(os.path.join(base_dir, folder, sub), exist_ok=True)

# UPDATED: Look for .jpeg instead of .jpg
files = [f.split('.')[0] for f in os.listdir(img_source) if f.endswith('.jpeg')]

if len(files) == 0:
    print("❌ Still no files found. Double check the extension in your folder!")
else:
    random.shuffle(files)
    split_idx = int(len(files) * 0.8)
    train_list = files[:split_idx]
    val_list = files[split_idx:]

    print(f"Transferring {len(train_list)} files to train and {len(val_list)} to val...")

    for name in train_list:
        # Match the .jpeg extension here too
        shutil.copy2(os.path.join(img_source, f"{name}.jpeg"), os.path.join(base_dir, 'train/images', f"{name}.jpg"))
        shutil.copy2(os.path.join(lbl_source, f"{name}.txt"), os.path.join(base_dir, 'train/labels', f"{name}.txt"))

    for name in val_list:
        shutil.copy2(os.path.join(img_source, f"{name}.jpeg"), os.path.join(base_dir, 'val/images', f"{name}.jpg"))
        shutil.copy2(os.path.join(lbl_source, f"{name}.txt"), os.path.join(base_dir, 'val/labels', f"{name}.txt"))

    print(f"✅ DONE! Train: {len(os.listdir(base_dir+'/train/images'))} | Val: {len(os.listdir(base_dir+'/val/images'))}")




Transferring 1040 files to train and 260 to val...
✅ DONE! Train: 1040 | Val: 260


In [11]:
yaml_text = """
train: /content/dataset/train/images
val: /content/dataset/val/images

nc: 2
names: ['crop', 'weed']
"""

with open('/content/data.yaml', 'w') as f:
    f.write(yaml_text)
print("✅ data.yaml created successfully!")

✅ data.yaml created successfully!


In [12]:
!pip install ultralytics -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 54.9 MB/s eta 0:00:00


In [13]:
from ultralytics import YOLO

# 1. Load the model
model = YOLO('yolov8n.pt')

# 2. Train the model
# We use imgsz=512 as per your project document [cite: 3, 12]
results = model.train(data='/content/data.yaml', epochs=25, imgsz=512)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.26 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=25, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=512, int8=False, i

In [15]:
from ultralytics import YOLO

# 1. Load your best trained model
model = YOLO('/content/runs/detect/train/weights/best.pt')

# 2. Run it on a validation image to create the visual proof
# This creates the 'runs/detect/predict' folder
results = model.predict(source='/content/dataset/val/images', save=True, imgsz=512, conf=0.25)

print("✅ Success! Go to 'runs/detect/predict' to see your labeled image.")


image 1/260 /content/dataset/val/images/agri_0_1017.jpg: 512x512 1 weed, 7.9ms
image 2/260 /content/dataset/val/images/agri_0_1020.jpg: 512x512 1 weed, 5.9ms
image 3/260 /content/dataset/val/images/agri_0_1024.jpg: 512x512 1 weed, 5.7ms
image 4/260 /content/dataset/val/images/agri_0_1068.jpg: 512x512 1 weed, 5.6ms
image 5/260 /content/dataset/val/images/agri_0_1083.jpg: 512x512 1 crop, 5.8ms
image 6/260 /content/dataset/val/images/agri_0_1095.jpg: 512x512 1 weed, 5.8ms
image 7/260 /content/dataset/val/images/agri_0_1119.jpg: 512x512 1 crop, 5.7ms
image 8/260 /content/dataset/val/images/agri_0_1123.jpg: 512x512 3 crops, 6.4ms
image 9/260 /content/dataset/val/images/agri_0_1146.jpg: 512x512 1 weed, 7.1ms
image 10/260 /content/dataset/val/images/agri_0_122.jpg: 512x512 1 weed, 5.6ms
image 11/260 /content/dataset/val/images/agri_0_126.jpg: 512x512 7 weeds, 5.5ms
image 12/260 /content/dataset/val/images/agri_0_1260.jpg: 512x512 2 crops, 5.9ms
image 13/260 /content/dataset/val/images/agri_0